# Practica 1 - ETL de Vuelos: de CSV crudo a Data Warehouse en SQL Server

**Seminario de Sistemas 2 - USAC**

Este notebook es el panel de control unico del proceso ETL: extrae el dataset crudo de vuelos, lo perfila y limpia, construye el modelo dimensional (esquema estrella) y lo carga a SQL Server, para finalmente validar la carga con las consultas analiticas del enunciado.

**Flujo:** Extraccion - Perfilado de calidad - Transformacion - Diagrama del modelo - Carga a SQL Server - Validacion (consultas analiticas)

Prerrequisitos: SQL Server corriendo localmente (instancia SQLEXPRESS), archivo `.env` configurado, dependencias instaladas (`pip install -r ../requirements.txt`).

In [77]:
import os
import urllib.parse
from pathlib import Path

import pandas as pd
import pyodbc
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(Path('..') / '.env')

DB_SERVER = os.getenv('DB_SERVER', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'VuelosDW')
DB_DRIVER = os.getenv('DB_DRIVER', 'ODBC Driver 18 for SQL Server')

DATA_PATH = Path('../data/dataset_vuelos_crudo.csv')
SCHEMA_SQL_PATH = Path('../sql/01_schema.sql')
QUERIES_SQL_PATH = Path('../sql/02_consultas_analiticas.sql')

pd.set_option('display.max_columns', 30)
print('Config cargada. Servidor:', DB_SERVER, '| Base:', DB_NAME)

Config cargada. Servidor: localhost\SQLEXPRESS | Base: VuelosDW


## 1. Extraccion

Se lee el archivo fuente tal cual llega, sin limpiar nada todavia, para dejar evidencia del estado crudo.

In [78]:
df_raw = pd.read_csv(DATA_PATH)
print('Filas:', df_raw.shape[0], '| Columnas:', df_raw.shape[1])
df_raw.info()

Filas: 10000 | Columnas: 26
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 26 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   record_id              10000 non-null  int64  
 1   airline_code           10000 non-null  str    
 2   airline_name           10000 non-null  str    
 3   flight_number          10000 non-null  str    
 4   origin_airport         10000 non-null  str    
 5   destination_airport    10000 non-null  str    
 6   departure_datetime     10000 non-null  str    
 7   arrival_datetime       9440 non-null   str    
 8   duration_min           9440 non-null   float64
 9   status                 10000 non-null  str    
 10  delay_min              9440 non-null   float64
 11  aircraft_type          10000 non-null  str    
 12  cabin_class            10000 non-null  str    
 13  seat                   9440 non-null   str    
 14  passenger_id           10000 non-null 

In [79]:
df_raw.head()

,record_id,airline_code,airline_name,flight_number,origin_airport,destination_airport,departure_datetime,arrival_datetime,duration_min,status,delay_min,aircraft_type,cabin_class,seat,passenger_id,passenger_gender,passenger_age,passenger_nationality,booking_datetime,sales_channel,payment_method,ticket_price,currency,ticket_price_usd_est,bags_total,bags_checked
0,1,FR,Ryanair,FR5515,MEX,GUA,20/01/2024 10:14,20/01/2024 13:24,185.0,ON_TIME,0.0,B739,ECONOMY,37F,77e21f8e-6e79-4504-905f-636cea932c06,M,40.0,PE,15/01/2024 10:09,APP,EFECTIVO,"77,60",USD,77.60,0,0
1,2,AV,Avianca,AV1170,SAP,jfk,22/12/2024 20:18,NaN,NaN,CANCELLED,NaN,CRJ9,ECONOMY,NaN,d44ad2a6-091a-49b9-841a-647f006c6f7b,M,42.0,CU,03/10/2024 08:42,APP,TARJETA,138.8,USD,138.80,1,0
2,3,IB,Iberia,IB2267,PTY,BOG,18/06/2025 12:16,18/06/2025 17:50,330.0,ON_TIME,0.0,A320,PREMIUM_ECONOMY,11B,4662a840-f754-403a-9eab-887d5f56b6f7,F,30.0,PE,14/05/2025 10:03,AEROPUERTO,TRANSFERENCIA,101.3,USD,101.30,1,0
3,4,DL,Delta,DL8318,HAV,MIA,12/06/2025 13:05,12/06/2025 17:20,261.0,ON_TIME,0.0,A319,ECONOMY,30E,3c53aed2-156e-44e1-9181-9e84437171dc,X,54.0,GT,03-15-2025 01:58 PM,AEROPUERTO,TRANSFERENCIA,106.55,GTQ,13.64,1,0
4,5,WN,Southwest,WN6538,GUA,SAL,04/10/2025 05:08,10/04/2025 08:59,210.0,ON_TIME,0.0,B767,BUSINESS,10C,273782b1-130d-4e4b-a947-b75769bfaeb5,M,27.0,CR,30/03/2025 02:22,CALL_CENTER,TARJETA,216.66,EUR,236.16,1,0


## 2. Perfilado de calidad de datos

Antes de transformar, se documentan los problemas reales encontrados en el dataset.

In [80]:
nulos = df_raw.isna().sum()
print('Columnas con valores nulos (de', len(df_raw), 'filas totales):')
print(nulos[nulos > 0])
print()
print('La mayoria de estos nulos son logicos, no errores de captura, son vuelos cancelados')

Columnas con valores nulos (de 10000 filas totales):
arrival_datetime         560
duration_min             560
delay_min                560
seat                     560
passenger_age            112
passenger_nationality    209
sales_channel            144
dtype: int64

La mayoria de estos nulos son logicos, no errores de captura, son vuelos cancelados


### Genero: valores inconsistentes

In [81]:
valores_genero = sorted(df_raw['passenger_gender'].dropna().unique())
print('Valores unicos de passenger_gender en el CSV crudo:')
print(valores_genero)
print()
print(f'Son {len(valores_genero)} valores de texto distintos para lo que logicamente')
print('deberian ser solo 3 categorias (masculino/femenino/no binario).')

Valores unicos de passenger_gender en el CSV crudo:
['F', 'Femenino', 'M', 'Masculino', 'NoBinario', 'X', 'f', 'femenino', 'm', 'masculino', 'nobinario', 'x']

Son 12 valores de texto distintos para lo que logicamente
deberian ser solo 3 categorias (masculino/femenino/no binario).


### Aeropuertos y aerolineas: casing inconsistente

In [82]:
def variantes_de_casing(serie):
    """Agrupa por version en MAYUSCULA y devuelve solo los casos donde el CSV
    trae mas de una forma de escribir el mismo valor (ej. 'JFK' y 'jfk')."""
    crudo = serie.dropna()
    agrupado = crudo.groupby(crudo.str.upper()).unique()
    return {clave: list(variantes) for clave, variantes in agrupado.items() if len(variantes) > 1}

aeropuertos_crudos = pd.concat([df_raw['origin_airport'], df_raw['destination_airport']])
print('Aeropuertos con mas de una variante de mayus/minuscula en la fuente:')
print(variantes_de_casing(aeropuertos_crudos))
print()
print('Aerolineas con mas de una variante de mayus/minuscula en la fuente:')
print(variantes_de_casing(df_raw['airline_name']))
print()
print('Total de textos distintos para aerolinea antes de normalizar:', df_raw['airline_name'].nunique())

Aeropuertos con mas de una variante de mayus/minuscula en la fuente:
{'BCN': ['BCN', 'bcn'], 'BOG': ['BOG', 'bog'], 'CUN': ['CUN', 'cun'], 'GUA': ['GUA', 'gua'], 'HAV': ['HAV', 'hav'], 'JFK': ['JFK', 'jfk'], 'LAX': ['LAX', 'lax'], 'LIM': ['LIM', 'lim'], 'MAD': ['MAD', 'mad'], 'MEX': ['MEX', 'mex'], 'MIA': ['MIA', 'mia'], 'PTY': ['PTY', 'pty'], 'SAL': ['SAL', 'sal'], 'SAP': ['SAP', 'sap'], 'SJO': ['SJO', 'sjo']}

Aerolineas con mas de una variante de mayus/minuscula en la fuente:
{'AEROMEXICO': ['AEROMEXICO', 'Aeromexico'], 'AMERICAN AIRLINES': ['AMERICAN AIRLINES', 'American Airlines'], 'AVIANCA': ['Avianca', 'AVIANCA'], 'BRITISH AIRWAYS': ['British Airways', 'BRITISH AIRWAYS'], 'COPA AIRLINES': ['Copa Airlines', 'COPA AIRLINES'], 'DELTA': ['Delta', 'DELTA'], 'IBERIA': ['Iberia', 'IBERIA'], 'JETBLUE': ['JetBlue', 'JETBLUE'], 'RYANAIR': ['Ryanair', 'RYANAIR'], 'SOUTHWEST': ['Southwest', 'SOUTHWEST'], 'UNITED': ['United', 'UNITED']}

Total de textos distintos para aerolinea antes de norm

### Fechas: formatos mezclados

In [83]:
formato_barra = df_raw['departure_datetime'].str.match(r'^\d{2}/\d{2}/\d{4} \d{2}:\d{2}$')
formato_guion = df_raw['departure_datetime'].str.match(r'^\d{2}-\d{2}-\d{4} \d{2}:\d{2} [AP]M$')

print('Formato DD/MM/YYYY HH:MM =', formato_barra.sum(), 'filas. Ejemplo:',
      df_raw.loc[formato_barra, 'departure_datetime'].iloc[0])
print('Formato MM-DD-YYYY HH:MM AM/PM =', formato_guion.sum(), 'filas. Ejemplo:',
      df_raw.loc[formato_guion, 'departure_datetime'].iloc[0])

Formato DD/MM/YYYY HH:MM = 8462 filas. Ejemplo: 20/01/2024 10:14
Formato MM-DD-YYYY HH:MM AM/PM = 1538 filas. Ejemplo: 12-27-2024 04:00 AM


### Precios, monedas y estados atipicos

In [84]:
con_coma = df_raw[df_raw['ticket_price'].astype(str).str.contains(',')]
sin_coma = df_raw[~df_raw['ticket_price'].astype(str).str.contains(',')]
print('Ejemplo con coma decimal (formato europeo):')
print(con_coma[['record_id', 'ticket_price', 'currency']].iloc[0].to_dict())
print('Ejemplo con punto decimal (formato estandar):')
print(sin_coma[['record_id', 'ticket_price', 'currency']].iloc[0].to_dict())
print('Total filas con coma decimal:', len(con_coma), 'de', len(df_raw))
print()
print('Estados de vuelo (incluye DIVERTED, no solo ON_TIME/DELAYED/CANCELLED):',
      sorted(df_raw['status'].dropna().unique()))
print('Filas sin sales_channel:',
      df_raw['sales_channel'].isna().sum(), 'de', len(df_raw))

Ejemplo con coma decimal (formato europeo):
{'record_id': 1, 'ticket_price': '77,60', 'currency': 'USD'}
Ejemplo con punto decimal (formato estandar):
{'record_id': 2, 'ticket_price': '138.8', 'currency': 'USD'}
Total filas con coma decimal: 930 de 10000

Estados de vuelo (incluye DIVERTED, no solo ON_TIME/DELAYED/CANCELLED): ['CANCELLED', 'DELAYED', 'DIVERTED', 'ON_TIME']
Filas sin sales_channel: 144 de 10000


## 3. Transformacion

Limpieza, normalizacion y construccion de dimensiones y hechos.

### 3.1 Fechas: doble formato con fallback

In [85]:
def parse_mixed_datetime(series):
    s = series.astype('string').str.strip()
    parsed = pd.to_datetime(s, format='%d/%m/%Y %H:%M', errors='coerce')
    faltantes = parsed.isna() & s.notna()
    parsed_fallback = pd.to_datetime(s[faltantes], format='%m-%d-%Y %I:%M %p', errors='coerce')
    parsed.loc[faltantes] = parsed_fallback
    return parsed

df = df_raw.copy()
df['departure_dt'] = parse_mixed_datetime(df['departure_datetime'])
df['arrival_dt'] = parse_mixed_datetime(df['arrival_datetime'])
df['booking_dt'] = parse_mixed_datetime(df['booking_datetime'])

sin_parsear = df['departure_dt'].isna().sum()
print(f'Fechas de salida sin poder parsear: {sin_parsear} de {len(df)}')

Fechas de salida sin poder parsear: 0 de 10000


### 3.2 Normalizacion de texto (aeropuertos, aerolineas, genero, catalogos)

In [86]:
df['origin_airport'] = df['origin_airport'].str.strip().str.upper()
df['destination_airport'] = df['destination_airport'].str.strip().str.upper()
df['airline_code'] = df['airline_code'].str.strip().str.upper()
df['airline_name'] = df['airline_name'].str.strip().str.title()

# columnas categoricas con nulos genuinos en la fuente (ej. sales_channel: 144 filas
# sin canal registrado) se completan con 'DESCONOCIDO' para no perder el registro
# completo del vuelo por un solo atributo faltante
df['sales_channel'] = df['sales_channel'].str.strip().str.upper().fillna('DESCONOCIDO')
df['payment_method'] = df['payment_method'].str.strip().str.upper().fillna('DESCONOCIDO')
df['cabin_class'] = df['cabin_class'].str.strip().str.upper().fillna('DESCONOCIDO')
df['status'] = df['status'].str.strip().str.upper()
df['passenger_nationality'] = df['passenger_nationality'].str.strip().str.upper()

def normalize_gender(valor):
    texto = str(valor).strip().lower()
    if texto in ('m', 'masculino'):
        return 'M'
    if texto in ('f', 'femenino'):
        return 'F'
    return 'X'

df['genero'] = df['passenger_gender'].apply(normalize_gender)
print('Generos normalizados:', df['genero'].value_counts().to_dict())

Generos normalizados: {'M': 4912, 'F': 4698, 'X': 390}


### 3.3 Dimension Fecha

In [87]:
MESES_ES = ['Enero', 'Febrero', 'Marzo', 'Abril', 'Mayo', 'Junio', 'Julio',
            'Agosto', 'Septiembre', 'Octubre', 'Noviembre', 'Diciembre']
DIAS_ES = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']

def construir_dim_fecha(*series_de_fechas):
    fechas = pd.concat(series_de_fechas).dropna().dt.normalize().unique()
    dim = pd.DataFrame({'fecha': pd.to_datetime(fechas)})
    dim['id_fecha'] = dim['fecha'].dt.strftime('%Y%m%d').astype(int)
    dim['anio'] = dim['fecha'].dt.year
    dim['mes'] = dim['fecha'].dt.month
    dim['nombre_mes'] = dim['mes'].apply(lambda m: MESES_ES[m - 1])
    dim['dia'] = dim['fecha'].dt.day
    dim['trimestre'] = dim['fecha'].dt.quarter
    dim['dia_semana'] = dim['fecha'].dt.weekday + 1
    dim['nombre_dia_semana'] = dim['fecha'].dt.weekday.apply(lambda d: DIAS_ES[d])
    dim['es_fin_de_semana'] = dim['dia_semana'].isin([6, 7]).astype(int)
    dim['fecha'] = dim['fecha'].dt.date
    columnas = ['id_fecha', 'fecha', 'anio', 'mes', 'nombre_mes', 'dia',
                'trimestre', 'dia_semana', 'nombre_dia_semana', 'es_fin_de_semana']
    return dim[columnas].sort_values('id_fecha').reset_index(drop=True)

dim_fecha = construir_dim_fecha(df['departure_dt'], df['booking_dt'])

df['id_fecha_salida'] = df['departure_dt'].dt.strftime('%Y%m%d').astype(int)
df['id_fecha_reserva'] = df['booking_dt'].dt.strftime('%Y%m%d').astype(int)

print('Fechas unicas en dim_fecha:', len(dim_fecha))
dim_fecha.head()

Fechas unicas en dim_fecha: 861


,id_fecha,fecha,anio,mes,nombre_mes,dia,trimestre,dia_semana,nombre_dia_semana,es_fin_de_semana
0,20230110,2023-01-10,2023,1,Enero,10,1,2,Martes,0
1,20230112,2023-01-12,2023,1,Enero,12,1,4,Jueves,0
2,20230210,2023-02-10,2023,2,Febrero,10,1,5,Viernes,0
3,20230211,2023-02-11,2023,2,Febrero,11,1,6,Sabado,1
4,20230212,2023-02-12,2023,2,Febrero,12,1,7,Domingo,1


### 3.4 Dimensiones de catalogo

In [88]:
dim_aerolinea = (
    df[['airline_code', 'airline_name']]
    .drop_duplicates(subset='airline_code')
    .rename(columns={'airline_code': 'codigo_aerolinea', 'airline_name': 'nombre_aerolinea'})
    .sort_values('codigo_aerolinea')
    .reset_index(drop=True)
)

aeropuertos_unicos = pd.concat([df['origin_airport'], df['destination_airport']]).dropna().unique()
dim_aeropuerto = pd.DataFrame({'codigo_iata': sorted(aeropuertos_unicos)})

dim_pasajero = (
    df[['passenger_id', 'genero', 'passenger_age', 'passenger_nationality']]
    .drop_duplicates(subset='passenger_id')
    .rename(columns={
        'passenger_id': 'passenger_uuid',
        'passenger_age': 'edad',
        'passenger_nationality': 'nacionalidad',
    })
    .reset_index(drop=True)
)

dim_canal_venta = pd.DataFrame({'canal': sorted(df['sales_channel'].dropna().unique())})
dim_pago = pd.DataFrame({'metodo_pago': sorted(df['payment_method'].dropna().unique())})
dim_cabina = pd.DataFrame({'clase_cabina': sorted(df['cabin_class'].dropna().unique())})

print('dim_aerolinea:', len(dim_aerolinea), '| dim_aeropuerto:', len(dim_aeropuerto),
      '| dim_pasajero:', len(dim_pasajero), '| dim_canal_venta:', len(dim_canal_venta),
      '| dim_pago:', len(dim_pago), '| dim_cabina:', len(dim_cabina))

dim_aerolinea: 12 | dim_aeropuerto: 15 | dim_pasajero: 10000 | dim_canal_venta: 6 | dim_pago: 5 | dim_cabina: 4


## 4. Modelo Dimensional

Esquema estrella con `hechos_vuelo` en el centro (ver `docs/modelo_dimensional.md` para la justificacion de cada decision de diseño):

![Modelo dimensional](../docs/modelo_dimensional.png)

## 5. Carga a SQL Server

Se conecta a la instancia local (SQLEXPRESS), se crea la base y las tablas con `sql/01_schema.sql`, se limpian datos de una corrida anterior (para que el notebook se pueda volver a correr en la presentacion sin errores de llave duplicada) y se cargan las dimensiones y luego los hechos.

In [89]:
def obtener_conexion_pyodbc(base_de_datos):
    conn_str = (
        f'DRIVER={{{DB_DRIVER}}};'
        f'SERVER={DB_SERVER};'
        f'DATABASE={base_de_datos};'
        f'Trusted_Connection=yes;'
        f'TrustServerCertificate=yes;'
    )
    return pyodbc.connect(conn_str, autocommit=True)

def crear_engine(base_de_datos):
    conn_str = (
        f'DRIVER={{{DB_DRIVER}}};'
        f'SERVER={DB_SERVER};'
        f'DATABASE={base_de_datos};'
        f'Trusted_Connection=yes;'
        f'TrustServerCertificate=yes;'
    )
    params = urllib.parse.quote_plus(conn_str)
    return create_engine(f'mssql+pyodbc:///?odbc_connect={params}', fast_executemany=True)

In [90]:
schema_sql = SCHEMA_SQL_PATH.read_text(encoding='utf-8')
batches = [b for b in schema_sql.split('\nGO') if b.strip()]

conn = obtener_conexion_pyodbc('master')
cur = conn.cursor()
for batch in batches:
    cur.execute(batch)
conn.close()
print(f'Ejecutados {len(batches)} batches de 01_schema.sql: base VuelosDW y tablas listas.')

Ejecutados 14 batches de 01_schema.sql: base VuelosDW y tablas listas.


In [91]:
engine = crear_engine(DB_NAME)

tablas_orden_borrado = [
    'hechos_vuelo', 'dim_fecha', 'dim_aerolinea', 'dim_aeropuerto',
    'dim_pasajero', 'dim_canal_venta', 'dim_pago', 'dim_cabina',
]
tablas_con_identity = [t for t in tablas_orden_borrado if t != 'dim_fecha']

with engine.begin() as conn:
    for tabla in tablas_orden_borrado:
        conn.execute(text(f'DELETE FROM {tabla}'))
    for tabla in tablas_con_identity:
        conn.execute(text(f"DBCC CHECKIDENT ('{tabla}', RESEED, 0)"))

print('Tablas limpiadas')

Tablas limpiadas


In [92]:
def cargar_dimension_con_identity(df_dim, nombre_tabla, columna_id, engine):
    df_dim.drop(columns=[columna_id], errors='ignore').to_sql(
        nombre_tabla, engine, if_exists='append', index=False
    )
    return pd.read_sql(f'SELECT * FROM {nombre_tabla}', engine)

dim_aerolinea = cargar_dimension_con_identity(dim_aerolinea, 'dim_aerolinea', 'id_aerolinea', engine)
dim_aeropuerto = cargar_dimension_con_identity(dim_aeropuerto, 'dim_aeropuerto', 'id_aeropuerto', engine)
dim_pasajero = cargar_dimension_con_identity(dim_pasajero, 'dim_pasajero', 'id_pasajero', engine)
dim_canal_venta = cargar_dimension_con_identity(dim_canal_venta, 'dim_canal_venta', 'id_canal', engine)
dim_pago = cargar_dimension_con_identity(dim_pago, 'dim_pago', 'id_pago', engine)
dim_cabina = cargar_dimension_con_identity(dim_cabina, 'dim_cabina', 'id_cabina', engine)

dim_fecha.to_sql('dim_fecha', engine, if_exists='append', index=False)

print('Dimensiones cargadas.')

Dimensiones cargadas.


### 5.1 Construccion de la tabla de hechos

Se hace *merge* del dataframe transformado contra cada dimension (ya con los IDs reales asignados por SQL Server) para resolver las llaves foraneas.

In [93]:
fact = df.merge(dim_aerolinea, left_on='airline_code', right_on='codigo_aerolinea', how='left')

fact = fact.merge(
    dim_aeropuerto.rename(columns={'id_aeropuerto': 'id_aeropuerto_origen', 'codigo_iata': 'origin_airport'}),
    on='origin_airport', how='left',
)
fact = fact.merge(
    dim_aeropuerto.rename(columns={'id_aeropuerto': 'id_aeropuerto_destino', 'codigo_iata': 'destination_airport'}),
    on='destination_airport', how='left',
)
fact = fact.merge(
    dim_pasajero[['id_pasajero', 'passenger_uuid']],
    left_on='passenger_id', right_on='passenger_uuid', how='left',
)
fact = fact.merge(
    dim_canal_venta.rename(columns={'canal': 'sales_channel'}), on='sales_channel', how='left',
)
fact = fact.merge(
    dim_pago.rename(columns={'metodo_pago': 'payment_method'}), on='payment_method', how='left',
)
fact = fact.merge(
    dim_cabina.rename(columns={'clase_cabina': 'cabin_class'}), on='cabin_class', how='left',
)

hechos_vuelo = pd.DataFrame({
    'record_id': fact['record_id'],
    'flight_number': fact['flight_number'],
    'seat': fact['seat'],
    'id_fecha_salida': fact['id_fecha_salida'],
    'id_fecha_reserva': fact['id_fecha_reserva'],
    'id_aerolinea': fact['id_aerolinea'],
    'id_aeropuerto_origen': fact['id_aeropuerto_origen'],
    'id_aeropuerto_destino': fact['id_aeropuerto_destino'],
    'id_pasajero': fact['id_pasajero'],
    'id_canal': fact['id_canal'],
    'id_pago': fact['id_pago'],
    'id_cabina': fact['id_cabina'],
    'status': fact['status'],
    'is_cancelled': (fact['status'] == 'CANCELLED').astype(int),
    'duration_min': fact['duration_min'],
    'delay_min': fact['delay_min'],
    'ticket_price_usd': fact['ticket_price_usd_est'].astype(float),
    'bags_total': fact['bags_total'],
    'bags_checked': fact['bags_checked'],
})

claves_foraneas = [
    'id_fecha_salida', 'id_fecha_reserva', 'id_aerolinea', 'id_aeropuerto_origen',
    'id_aeropuerto_destino', 'id_pasajero', 'id_canal', 'id_pago', 'id_cabina',
]
nulos_en_fk = hechos_vuelo[claves_foraneas].isna().sum().sum()
assert nulos_en_fk == 0, f'Hay {nulos_en_fk} llaves foraneas sin resolver, revisar el merge.'
print('Todas las llaves foraneas se resolvieron correctamente.')
print('Filas listas para cargar en hechos_vuelo:', len(hechos_vuelo))

Todas las llaves foraneas se resolvieron correctamente.
Filas listas para cargar en hechos_vuelo: 10000


In [94]:
hechos_vuelo.to_sql('hechos_vuelo', engine, if_exists='append', index=False, chunksize=1000)
total_cargado = pd.read_sql('SELECT COUNT(*) AS total FROM hechos_vuelo', engine)['total'][0]
print('Carga completa: hechos_vuelo tiene', total_cargado, 'filas en SQL Server.')

Carga completa: hechos_vuelo tiene 10000 filas en SQL Server.


## 6. Validacion - Consultas Analiticas

Se ejecutan directamente contra SQL Server las consultas de `sql/02_consultas_analiticas.sql` y se muestran los resultados como evidencia de que la carga fue correcta.

In [95]:
from IPython.display import Markdown

queries_sql = QUERIES_SQL_PATH.read_text(encoding='utf-8')
batches_consultas = [b.strip() for b in queries_sql.split('\nGO') if b.strip()]

for batch in batches_consultas:
    if 'SELECT' not in batch.upper():
        continue
    lineas = batch.splitlines()
    titulo = next((l.lstrip('-- ').strip() for l in lineas if l.strip().startswith('--')), 'Consulta')
    display(Markdown(f'#### {titulo}'))
    resultado = pd.read_sql(text(batch), engine)
    display(resultado)

#### 1) Validacion de carga: total de registros en cada tabla

,tabla,filas
0,dim_fecha,861
1,dim_aerolinea,12
2,dim_aeropuerto,15
3,dim_pasajero,10000
4,dim_canal_venta,6
5,dim_pago,5
6,dim_cabina,4
7,hechos_vuelo,10000


#### 2) Numero total de vuelos cargados

,total_vuelos
0,10000


#### 3) Top 5 destinos mas frecuentes

,destino,cantidad_vuelos
0,SAP,701
1,CUN,699
2,BCN,696
3,BOG,696
4,HAV,693


... (10 outputs left out)

#### 9) Retraso promedio por aeropuerto de origen, solo vuelos no cancelados

,origen,vuelos,retraso_promedio_min
0,PTY,632,31.07
1,MIA,673,29.66
2,SJO,641,27.80
3,LAX,590,27.56
4,SAP,597,27.27
5,SAL,664,27.10
6,LIM,615,27.07
7,MEX,602,26.58
8,JFK,637,25.45
9,BOG,626,24.85


#### 10) Vuelos por mes

,anio,mes,nombre_mes,cantidad_vuelos
0,2024,1,Enero,423
1,2024,2,Febrero,380
2,2024,3,Marzo,416
3,2024,4,Abril,420
4,2024,5,Mayo,402
5,2024,6,Junio,370
6,2024,7,Julio,404
7,2024,8,Agosto,428
8,2024,9,Septiembre,391
9,2024,10,Octubre,464


## Cierre

El notebook dejo cargado el modelo dimensional completo en `VuelosDW` (SQL Server) y valido la carga con las consultas analiticas de arriba. Para la presentacion:

- Este notebook puede correrse de nuevo en cualquier momento, limpia y recarga sin errores.
- `sql/01_schema.sql` y `sql/02_consultas_analiticas.sql` son deliverables independientes, y tambien se pueden abrir/correr en SSMS como respaldo.
- `docs/modelo_dimensional.md` documenta cada decision de limpieza y diseno con mas detalle.